# EuRoC Chebyshev Spectrogram

This notebook fits one spectral Chebyshev polynomial per one-second interval in a merged EuRoC CSV. It uses `gtsam.Chebyshev1Basis`, where `N` is the number of weighted spectral basis functions and the polynomial degree is `n = N - 1`.

The fit is component-wise: for a selected signal group with `d` columns, every one-second interval gets an `N x d` coefficient matrix. The notebook displays easy and aggressive intervals, their spectral weights, an `m x N` coefficient spectrogram, and average spectra for the whole trajectory and trajectory parts.

This intentionally does **not** use `gtsam.Chebyshev2` pseudo-spectral node values. Each window is normalized to `tau in [-1, 1]`, then fitted with ordinary weighted Chebyshev basis functions `T_0(tau), ..., T_{N-1}(tau)`.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import clear_output, display
from plotly.subplots import make_subplots

import gtsam

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except Exception as exc:  # pragma: no cover - notebook convenience fallback
    HAS_WIDGETS = False
    WIDGET_IMPORT_ERROR = exc

pd.options.display.precision = 6

In [ ]:
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data" / "euroc").exists():
    REPO_ROOT = Path("/Users/dellaert/git/imuFactors")

DATA_DIR = REPO_ROOT / "data" / "euroc"
DATA_FILES = sorted(DATA_DIR.glob("euroc_*.csv"))
if not DATA_FILES:
    raise FileNotFoundError(f"No EuRoC CSV files found in {DATA_DIR}")

DEFAULT_FILE = DATA_DIR / "euroc_MH01.csv"
if not DEFAULT_FILE.exists():
    DEFAULT_FILE = DATA_FILES[0]

DEFAULT_N = 16  # degree n = N - 1
DEFAULT_SIGNAL_GROUP = "imu"
WINDOW_SECONDS = 1.0

SIGNAL_GROUPS = {
    "imu": ["w_x", "w_y", "w_z", "a_x", "a_y", "a_z"],
    "gyro": ["w_x", "w_y", "w_z"],
    "accel": ["a_x", "a_y", "a_z"],
    "position": ["p_x", "p_y", "p_z"],
    "velocity": ["v_x", "v_y", "v_z"],
    "gyro_bias": ["b_w_x", "b_w_y", "b_w_z"],
    "accel_bias": ["b_a_x", "b_a_y", "b_a_z"],
    "quaternion_components": ["q_w", "q_x", "q_y", "q_z"],
    "state_plus_bias": [
        "q_w", "q_x", "q_y", "q_z",
        "v_x", "v_y", "v_z",
        "p_x", "p_y", "p_z",
        "b_w_x", "b_w_y", "b_w_z",
        "b_a_x", "b_a_y", "b_a_z",
    ],
    "all_numeric": [],  # filled from the selected dataframe
}

print(f"Found {len(DATA_FILES)} EuRoC CSV files in {DATA_DIR}")

In [ ]:
@dataclass
class ChebyshevFitResult:
    path: Path
    N: int
    columns: list[str]
    dataframe: pd.DataFrame
    time: np.ndarray
    starts: np.ndarray
    steps_per_window: int
    sample_count: int
    window_seconds: float
    dt: float
    tau: np.ndarray
    weight_matrix: np.ndarray
    coeffs: np.ndarray
    coeffs_standardized: np.ndarray
    coeff_energy: np.ndarray
    samples: np.ndarray
    reconstructed: np.ndarray
    rmse: np.ndarray
    center: np.ndarray
    scale: np.ndarray
    activity: np.ndarray
    high_order_ratio: np.ndarray


def load_euroc_csv(path: str | Path) -> pd.DataFrame:
    """Load a merged EuRoC CSV and make quaternion signs continuous."""
    df = pd.read_csv(path)
    quat_cols = ["q_w", "q_x", "q_y", "q_z"]
    if all(column in df.columns for column in quat_cols):
        quat = df[quat_cols].to_numpy(dtype=float, copy=True)
        for i in range(1, len(quat)):
            if float(np.dot(quat[i - 1], quat[i])) < 0.0:
                quat[i] *= -1.0
        df.loc[:, quat_cols] = quat
    return df


def available_signal_groups(df: pd.DataFrame) -> dict[str, list[str]]:
    groups: dict[str, list[str]] = {}
    for name, columns in SIGNAL_GROUPS.items():
        if name == "all_numeric":
            continue
        present = [column for column in columns if column in df.columns]
        if present:
            groups[name] = present
    groups["all_numeric"] = [
        column for column in df.columns
        if column != "t" and pd.api.types.is_numeric_dtype(df[column])
    ]
    return groups


def robust_center_scale(values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    center = np.nanmedian(values, axis=0)
    mad = np.nanmedian(np.abs(values - center), axis=0)
    robust_sigma = 1.4826 * mad
    standard_sigma = np.nanstd(values, axis=0)
    scale = np.where(robust_sigma > 1e-12, robust_sigma, standard_sigma)
    scale = np.where(scale > 1e-12, scale, 1.0)
    return center, scale


def one_second_window_starts(
    time: np.ndarray,
    window_seconds: float = WINDOW_SECONDS,
) -> tuple[np.ndarray, int, float]:
    dt = float(np.median(np.diff(time)))
    steps_per_window = int(round(window_seconds / dt))
    if steps_per_window < 1:
        raise ValueError("Window duration is shorter than one sample interval")
    starts = np.arange(0, len(time) - steps_per_window, steps_per_window)
    durations = time[starts + steps_per_window] - time[starts]
    tolerance = max(1e-6, 0.1 * dt)
    starts = starts[np.abs(durations - window_seconds) <= tolerance]
    if len(starts) == 0:
        raise ValueError("No complete one-second windows found")
    return starts, steps_per_window, dt


def chebyshev1_design(N: int, sample_count: int) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return canonical tau samples, spectral Chebyshev weights, and LS solver."""
    if N < 1:
        raise ValueError("N must be positive")
    if N > sample_count:
        raise ValueError(f"N={N} exceeds the {sample_count} samples in a window")
    tau = np.linspace(-1.0, 1.0, sample_count)
    weight_matrix = np.asarray(gtsam.Chebyshev1Basis.WeightMatrix(int(N), tau), dtype=float)
    solver = np.linalg.pinv(weight_matrix)
    return tau, weight_matrix, solver


def fit_chebyshev_windows(
    path: str | Path,
    N: int,
    signal_group: str = DEFAULT_SIGNAL_GROUP,
    columns: list[str] | None = None,
    window_seconds: float = WINDOW_SECONDS,
) -> ChebyshevFitResult:
    path = Path(path)
    df = load_euroc_csv(path)
    groups = available_signal_groups(df)
    if columns is None:
        if signal_group not in groups:
            raise KeyError(f"Unknown signal group {signal_group!r}; available: {sorted(groups)}")
        columns = groups[signal_group]
    columns = [column for column in columns if column in df.columns]
    if not columns:
        raise ValueError("No selected columns are present in the dataframe")

    time = df["t"].to_numpy(dtype=float)
    starts, steps_per_window, dt = one_second_window_starts(time, window_seconds)
    sample_count = steps_per_window + 1
    tau, weight_matrix, solver = chebyshev1_design(int(N), sample_count)

    values = df[columns].to_numpy(dtype=float)
    center, scale = robust_center_scale(values)

    m = len(starts)
    d = len(columns)
    samples = np.empty((m, sample_count, d), dtype=float)
    reconstructed = np.empty_like(samples)
    coeffs = np.empty((m, int(N), d), dtype=float)

    for window_index, start in enumerate(starts):
        stop = start + sample_count
        y = values[start:stop, :]
        coefficient_matrix = solver @ y
        samples[window_index] = y
        coeffs[window_index] = coefficient_matrix
        reconstructed[window_index] = weight_matrix @ coefficient_matrix

    residual = reconstructed - samples
    rmse = np.sqrt(np.mean(residual * residual, axis=1))

    coeffs_standardized = coeffs / scale.reshape(1, 1, -1)
    coeffs_standardized[:, 0, :] -= (center / scale).reshape(1, -1)
    coeff_energy = np.sqrt(np.mean(coeffs_standardized * coeffs_standardized, axis=2))

    standardized_samples = (samples - center.reshape(1, 1, -1)) / scale.reshape(1, 1, -1)
    activity = np.sqrt(np.mean(np.var(standardized_samples, axis=1), axis=1))
    high_start = max(2, int(N) // 2)
    high_energy = np.sum(coeff_energy[:, high_start:], axis=1)
    non_dc_energy = np.sum(coeff_energy[:, 1:], axis=1)
    high_order_ratio = high_energy / np.maximum(non_dc_energy, 1e-12)

    return ChebyshevFitResult(
        path=path,
        N=int(N),
        columns=list(columns),
        dataframe=df,
        time=time,
        starts=starts,
        steps_per_window=steps_per_window,
        sample_count=sample_count,
        window_seconds=window_seconds,
        dt=dt,
        tau=tau,
        weight_matrix=weight_matrix,
        coeffs=coeffs,
        coeffs_standardized=coeffs_standardized,
        coeff_energy=coeff_energy,
        samples=samples,
        reconstructed=reconstructed,
        rmse=rmse,
        center=center,
        scale=scale,
        activity=activity,
        high_order_ratio=high_order_ratio,
    )


def zscore(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=float)
    sigma = float(np.nanstd(values))
    if sigma <= 1e-12:
        return np.zeros_like(values)
    return (values - float(np.nanmean(values))) / sigma


def normalized_rmse(result: ChebyshevFitResult) -> np.ndarray:
    return np.sqrt(np.mean((result.rmse / result.scale.reshape(1, -1)) ** 2, axis=1))


def characteristic_windows(result: ChebyshevFitResult) -> dict[str, int]:
    easy_score = zscore(result.activity) + zscore(result.high_order_ratio) + zscore(normalized_rmse(result))
    aggressive_score = zscore(result.activity) + 1.5 * zscore(result.high_order_ratio)
    easy = int(np.nanargmin(easy_score))
    aggressive = int(np.nanargmax(aggressive_score))
    if aggressive == easy and len(result.starts) > 1:
        aggressive = int(np.nanargmax(result.activity))
    return {"easy": easy, "aggressive": aggressive}


def window_start_seconds(result: ChebyshevFitResult, window_index: int) -> float:
    return float(result.time[result.starts[window_index]] - result.time[0])

In [ ]:
def summary_table(result: ChebyshevFitResult) -> pd.DataFrame:
    duration = float(result.time[-1] - result.time[0])
    rows = [
        ("file", result.path.name),
        ("selected columns", ", ".join(result.columns)),
        ("N", result.N),
        ("polynomial degree", result.N - 1),
        ("complete one-second windows", len(result.starts)),
        ("samples per closed window", result.sample_count),
        ("median dt", result.dt),
        ("effective rate", 1.0 / result.dt),
        ("file duration", duration),
    ]
    return pd.DataFrame(rows, columns=["quantity", "value"])


def interval_metrics_table(result: ChebyshevFitResult, selected: dict[str, int]) -> pd.DataFrame:
    nrms = normalized_rmse(result)
    rows = []
    for label, index in selected.items():
        rows.append({
            "label": label,
            "window_index": index,
            "start_s": window_start_seconds(result, index),
            "activity": result.activity[index],
            "high_order_ratio": result.high_order_ratio[index],
            "normalized_rmse": nrms[index],
        })
    return pd.DataFrame(rows)


def plot_metric_scatter(result: ChebyshevFitResult, selected: dict[str, int]) -> go.Figure:
    starts_s = np.array([window_start_seconds(result, i) for i in range(len(result.starts))])
    nrms = normalized_rmse(result)
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=result.activity,
        y=result.high_order_ratio,
        mode="markers",
        text=[f"window {i}, t={starts_s[i]:.1f}s" for i in range(len(result.starts))],
        marker=dict(size=8, color=nrms, colorscale="Turbo", showscale=True, colorbar_title="norm RMSE"),
        name="all windows",
    ))
    marker_symbols = {"easy": "circle-open", "aggressive": "x"}
    for label, index in selected.items():
        fig.add_trace(go.Scatter(
            x=[result.activity[index]],
            y=[result.high_order_ratio[index]],
            mode="markers+text",
            text=[label],
            textposition="top center",
            marker=dict(size=14, symbol=marker_symbols.get(label, "diamond"), color="black", line=dict(width=2)),
            name=label,
        ))
    fig.update_layout(
        title="Window characteristics used to choose easy and aggressive examples",
        xaxis_title="standardized within-window activity",
        yaxis_title="high-order coefficient energy ratio",
        height=450,
        margin=dict(l=70, r=40, t=70, b=60),
    )
    return fig


def plot_interval_fit(
    result: ChebyshevFitResult,
    window_index: int,
    label: str,
    max_components: int = 6,
) -> go.Figure:
    component_count = min(max_components, len(result.columns))
    columns = result.columns[:component_count]
    seconds = np.linspace(0.0, result.window_seconds, result.sample_count)
    fig = make_subplots(
        rows=component_count,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.025,
        subplot_titles=columns,
    )
    for row, column in enumerate(columns, start=1):
        component = result.columns.index(column)
        fig.add_trace(
            go.Scatter(
                x=seconds,
                y=result.samples[window_index, :, component],
                mode="markers",
                marker=dict(size=4),
                name=f"{column} samples",
                legendgroup=column,
            ),
            row=row,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=seconds,
                y=result.reconstructed[window_index, :, component],
                mode="lines",
                line=dict(width=2),
                name=f"{column} Cheb fit",
                legendgroup=column,
            ),
            row=row,
            col=1,
        )
        fig.update_yaxes(title_text=column, row=row, col=1)
    start_s = window_start_seconds(result, window_index)
    fig.update_xaxes(title_text="seconds inside window", row=component_count, col=1)
    fig.update_layout(
        title=f"{label.title()} interval fit: window {window_index}, start {start_s:.1f}s",
        height=max(320, 180 * component_count),
        margin=dict(l=70, r=30, t=80, b=60),
    )
    return fig


def plot_interval_coefficients(
    result: ChebyshevFitResult,
    window_index: int,
    label: str,
    standardized: bool = False,
) -> go.Figure:
    matrix = result.coeffs_standardized[window_index].T if standardized else result.coeffs[window_index].T
    title_prefix = "standardized" if standardized else "raw"
    zmax = float(np.nanmax(np.abs(matrix))) if matrix.size else 1.0
    if zmax <= 0.0:
        zmax = 1.0
    fig = go.Figure(data=go.Heatmap(
        z=matrix,
        x=[f"T{k}" for k in range(result.N)],
        y=result.columns,
        colorscale="RdBu",
        zmid=0.0,
        zmin=-zmax,
        zmax=zmax,
        colorbar_title="weight",
    ))
    start_s = window_start_seconds(result, window_index)
    fig.update_layout(
        title=f"{label.title()} interval {title_prefix} spectral weights: window {window_index}, start {start_s:.1f}s",
        xaxis_title="Chebyshev spectral basis",
        yaxis_title="signal component",
        height=max(360, 26 * len(result.columns) + 160),
        margin=dict(l=120, r=40, t=80, b=60),
    )
    return fig


def plot_coeff_spectrogram(result: ChebyshevFitResult, log_scale: bool = True) -> go.Figure:
    z = result.coeff_energy
    colorbar_title = "standardized RMS weight"
    if log_scale:
        z = np.log10(z + 1e-12)
        colorbar_title = "log10 standardized RMS weight"
    starts_s = np.array([window_start_seconds(result, i) for i in range(len(result.starts))])
    fig = go.Figure(data=go.Heatmap(
        z=z,
        x=[f"T{k}" for k in range(result.N)],
        y=starts_s,
        colorscale="Viridis",
        colorbar_title=colorbar_title,
    ))
    fig.update_layout(
        title="Coefficient spectrogram (m one-second intervals x N basis weights)",
        xaxis_title="Chebyshev spectral basis",
        yaxis_title="window start time [s]",
        height=max(420, min(900, 220 + 3 * len(result.starts))),
        margin=dict(l=80, r=40, t=80, b=60),
    )
    return fig


def plot_average_spectra(result: ChebyshevFitResult, part_count: int = 4) -> go.Figure:
    basis = np.arange(result.N)
    energy = result.coeff_energy
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=basis,
        y=np.mean(energy, axis=0),
        mode="lines+markers",
        line=dict(width=4, color="black"),
        name="whole file",
    ))
    edges = np.linspace(0, len(result.starts), part_count + 1, dtype=int)
    for part in range(part_count):
        a, b = int(edges[part]), int(edges[part + 1])
        if b <= a:
            continue
        start_s = window_start_seconds(result, a)
        end_s = window_start_seconds(result, b - 1) + result.window_seconds
        fig.add_trace(go.Scatter(
            x=basis,
            y=np.mean(energy[a:b], axis=0),
            mode="lines+markers",
            name=f"part {part + 1}: {start_s:.0f}-{end_s:.0f}s",
        ))
    fig.update_layout(
        title="Average standardized coefficient spectra",
        xaxis_title="Chebyshev spectral basis index k",
        yaxis_title="mean RMS coefficient energy across selected components",
        height=450,
        margin=dict(l=80, r=40, t=80, b=60),
    )
    return fig




def plot_average_spectrogram_parts(result: ChebyshevFitResult, part_count: int = 4) -> go.Figure:
    energy = result.coeff_energy
    rows = [np.mean(energy, axis=0)]
    labels = ["whole file"]
    edges = np.linspace(0, len(result.starts), part_count + 1, dtype=int)
    for part in range(part_count):
        a, b = int(edges[part]), int(edges[part + 1])
        if b <= a:
            continue
        start_s = window_start_seconds(result, a)
        end_s = window_start_seconds(result, b - 1) + result.window_seconds
        rows.append(np.mean(energy[a:b], axis=0))
        labels.append(f"part {part + 1}: {start_s:.0f}-{end_s:.0f}s")
    z = np.log10(np.vstack(rows) + 1e-12)
    fig = go.Figure(data=go.Heatmap(
        z=z,
        x=[f"T{k}" for k in range(result.N)],
        y=labels,
        colorscale="Viridis",
        colorbar_title="log10 mean RMS weight",
    ))
    fig.update_layout(
        title="Average coefficient spectrograms: whole file and trajectory parts",
        xaxis_title="Chebyshev spectral basis",
        yaxis_title="trajectory section",
        height=360,
        margin=dict(l=130, r=40, t=80, b=60),
    )
    return fig

def run_dashboard(
    path: str | Path = DEFAULT_FILE,
    N: int = DEFAULT_N,
    signal_group: str = DEFAULT_SIGNAL_GROUP,
    max_components: int = 6,
    show: bool = True,
) -> ChebyshevFitResult:
    result = fit_chebyshev_windows(path, int(N), signal_group=signal_group)
    selected = characteristic_windows(result)
    if show:
        display(summary_table(result))
        display(interval_metrics_table(result, selected))
        plot_metric_scatter(result, selected).show()
        for label, window_index in selected.items():
            plot_interval_fit(result, window_index, label, max_components=max_components).show()
            plot_interval_coefficients(result, window_index, label, standardized=False).show()
        plot_coeff_spectrogram(result, log_scale=True).show()
        plot_average_spectra(result, part_count=4).show()
        plot_average_spectrogram_parts(result, part_count=4).show()
    return result

## Interactive Controls

Use the controls below to choose the merged EuRoC CSV, the spectral coefficient count `N`, and the signal group. `N = n + 1`, so `N=16` fits a degree-15 polynomial in every complete one-second interval.

Adjacent windows share the endpoint sample: at 200 Hz, each closed one-second window contains 201 samples, and starts advance by 200 samples. The short trailing remainder is ignored.

In [ ]:
def make_dashboard_controls() -> tuple[Any, Any] | tuple[None, None]:
    if not HAS_WIDGETS:
        print("ipywidgets is not available; edit DEFAULT_FILE, DEFAULT_N, and DEFAULT_SIGNAL_GROUP manually.")
        print(f"Widget import error: {WIDGET_IMPORT_ERROR!r}")
        return None, None

    file_dropdown = widgets.Dropdown(
        options=[(path.name, str(path)) for path in DATA_FILES],
        value=str(DEFAULT_FILE),
        description="file",
        layout=widgets.Layout(width="520px"),
    )
    n_slider = widgets.IntSlider(
        value=DEFAULT_N,
        min=2,
        max=80,
        step=1,
        description="N",
        continuous_update=False,
        layout=widgets.Layout(width="520px"),
    )
    group_dropdown = widgets.Dropdown(
        options=list(SIGNAL_GROUPS.keys()),
        value=DEFAULT_SIGNAL_GROUP,
        description="signals",
        layout=widgets.Layout(width="520px"),
    )
    component_slider = widgets.IntSlider(
        value=6,
        min=1,
        max=12,
        step=1,
        description="plot dims",
        continuous_update=False,
        layout=widgets.Layout(width="520px"),
    )
    run_button = widgets.Button(description="Run fit", button_style="primary", icon="play")
    output = widgets.Output()

    def on_run(_: Any) -> None:
        with output:
            clear_output(wait=True)
            global LAST_RESULT
            LAST_RESULT = run_dashboard(
                Path(file_dropdown.value),
                int(n_slider.value),
                signal_group=str(group_dropdown.value),
                max_components=int(component_slider.value),
                show=True,
            )

    run_button.on_click(on_run)
    controls = widgets.VBox([
        widgets.HBox([file_dropdown]),
        widgets.HBox([n_slider]),
        widgets.HBox([group_dropdown]),
        widgets.HBox([component_slider, run_button]),
    ])
    return controls, output


controls, output = make_dashboard_controls()
if controls is not None:
    display(controls, output)
    with output:
        LAST_RESULT = run_dashboard(DEFAULT_FILE, DEFAULT_N, DEFAULT_SIGNAL_GROUP, max_components=6)
else:
    LAST_RESULT = run_dashboard(DEFAULT_FILE, DEFAULT_N, DEFAULT_SIGNAL_GROUP, max_components=6)

## Notes

- The coefficient heatmaps labeled `raw spectral weights` are the actual least-squares coefficients multiplying `T_k(tau)` for each selected signal component.
- The spectrogram and average spectra use robustly standardized coefficients so that components with different units can be combined into one energy image. The raw coefficients remain available in `LAST_RESULT.coeffs` with shape `(m, N, d)`.
- Quaternion columns are fitted component-wise after sign-continuity correction. This is useful for spectral inspection, but it is not a manifold-constrained attitude fit.